# 矩阵计算+自动求导

## 矩阵计算

基础知识：
1. 导数是切线的斜率
2. 亚导数，导数拓展到不可微分的函数
3. 梯度：将导数拓展到向量

梯度和等高线正交，指向值变化最大的方向

求**标量和向量**（假设列向量）的导数时，向量在分子结果还是**列向量**，向量在分母结果是**行向量**

求**向量和向量**的导数结果是**矩阵**，由上可知，分子相当于列向量，分母是行向量，列×行

## 自动求导

链式法则：
dy/dx = dy/du * du/dx 

计算图是数据结构图，通过多个节点和边表示
- 显示构造：Tensorflow/Theano/MXNet
- 隐式构造：PyTorch/MXNet

自动求导计算一个函数在指定值上的导数
1. 正向累积
2. 反向传递

In [1]:
import torch
x=torch.arange(4.0)
x

tensor([0., 1., 2., 3.])

In [ ]:
x.requires_grad_(True)   #下划线表示重写内容
x.grad           #将梯度存贮在x.grad中

In [ ]:
y = 2 * torch.dot(x, x)   #y=2x^2
y

tensor(28., grad_fn=<MulBackward0>)

In [ ]:
y.backward()   #反向求导数，计算梯度
x.grad

tensor([ 0.,  4.,  8., 12.])

In [6]:
x.grad == 4*x

tensor([True, True, True, True])

In [ ]:
#默认情况，pytorch会累积梯度，因此在反向传播之前需要清除梯度
x.grad.zero_()  #清除梯度
y=x.sum()
y.backward()
x.grad

tensor([1., 1., 1., 1.])

In [9]:
x

tensor([0., 1., 2., 3.], requires_grad=True)

In [ ]:
# 对非标量调⽤backward需要传⼊⼀个gradient参数，该参数指定微分函数关于self的梯度。
x.grad.zero_()
y = x * x     #hadamard积，逐元素相乘，x^2
y.sum().backward()
x.grad   #2x

tensor([0., 2., 4., 6.])

分离操作，将某些计算移动到记录的计算图之外

In [ ]:
x.grad.zero_()  #清除梯度
y = x * x     #hadamard积，逐元素相乘，x^2
u=y.detach()  #分离计算图，u不再需要梯度
z=u*x         #u作为常数参与计算
z.sum().backward()
x.grad == u        #证明成功分离u

tensor([True, True, True, True])

In [12]:
x.grad.zero_()  #清除梯度
y.sum().backward()
x.grad == 2*x

tensor([True, True, True, True])

In [ ]:
# 对于任何a，存在某个常量标量k，使得f(a)=k*a，其中k的值取决于输⼊a
def f(a):
    b = a * 2
    while b.norm() < 1000:
        b = b * 2
    if b.sum() > 0:
        c = b
    else:
        c = 100 * b
    return c

In [ ]:
# a是标量
a = torch.randn(size=(), requires_grad=True)
d = f(a)
d.backward()

In [ ]:
# d始终是关于a的线性（正比例）函数
a.grad == d/a

tensor(True)